In [3]:
import json
import paho.mqtt.client as mqtt
from load_model import load_best_model
from digital_twin import DigitalTwin

# Load the pre-trained model
model = load_best_model('best_entropy_model.pkl')
digital_twin = DigitalTwin(model)

# MQTT settings
BROKER = "broker.hivemq.com"
PORT = 1883
TOPIC = "patient/data"

# Callback when a message is received
def on_message(client, userdata, message):
    patient_data = json.loads(message.payload.decode())
    print(f"Received data: {patient_data}")

    # Predict patient health status
    patient_id = patient_data.pop('id')
    data_values = list(patient_data.values())
    prediction, probability = digital_twin.predict_health_status(data_values)

    # Display prediction
    print(f"Patient {patient_id} Prediction: {'Critical' if prediction == 1 else 'Healthy'} (Probability: {probability:.2f})")

    # Trigger emergency alert if necessary
    if prediction == 1 and probability > 0.9:
        digital_twin.alert_emergency_services(patient_id)

# MQTT Client setup
client = mqtt.Client()
client.on_message = on_message
client.connect(BROKER, PORT)
client.subscribe(TOPIC)

print("Listening for patient data...")
client.loop_forever()


ModuleNotFoundError: No module named 'load_model'